### MBA FP-gorwth algorithm to geenerate new bundles from most purschased bundles togather

In [ ]:
import pandas as pd
import os

In [6]:
DATA_PATH: str = r"C:\Users\Alber\Desktop\uni\GraduationProject\datasets\daily subscriptions"
FILEs: str = [rf'{DATA_PATH}\daily_sub_202508_with_locations_mobiles.csv',
            rf'{DATA_PATH}\daily_sub_202509_with_locations_mobiles.csv',
            rf'{DATA_PATH}\daily_sub_202510_with_locations_mobiles.csv'
]
SAVE_TO: str = r"C:\Users\Alber\Desktop\uni\GraduationProject\datasets\weekly purchases"

In [23]:
data = pd.read_csv(FILEs[0], 
    usecols=['msisdn', 'tbl_dt', 'bundle_name', 'bundle_id', 'total_rev', 'subscriptions']
)
data = data[(data['total_rev'] > 0) & (data['subscriptions'] > 0)]
data.shape

(45985969, 6)

In [24]:
data.head(10)

,tbl_dt,msisdn,bundle_id,bundle_name,subscriptions,total_rev
0,20250809,242069835561,SMSPK50,SMS Bundle 2 Days @66F,1,66.0
1,20250809,242064381075,FVX1D200FV3,Forfait Maxivoice 16Mins 1 jour@200F,1,200.0
2,20250809,242067637060,FVX1D200FV3,Forfait Maxivoice 16Mins 1 jour@200F,2,400.0
3,20250809,242067907977,SMSPK50,SMS Bundle 2 Days @66F,1,66.0
4,20250809,242061299388,36772,Whatsapp txt 1 jour@100F,1,100.0
5,20250809,242065583899,36769,Forfait Maximise 3.5GB 1 jour + Free Tik tok@1...,1,1300.0
6,20250821,242064022474,37249,Inbound Data Acceleration 100MB 1 day@100F,1,100.0
7,20250821,242064115736,36000,Forfait DIY VOICE ONNET,1,135.0
8,20250821,242064421850,36000,Forfait DIY VOICE ONNET,1,180.0
9,20250821,242065963316,SMSPK25,SMS Bundle 1 Day @41F,1,41.0


In [25]:
data['tbl_dt'] = pd.to_datetime(data['tbl_dt'], format='%Y%m%d')
data['YEAR'] = data['tbl_dt'].dt.year.astype(int)

data['WEEKNUMBER'] = data['tbl_dt'].dt.isocalendar().week

In [26]:
weekly_purchases = (
    data
    .groupby(['msisdn', 'WEEKNUMBER', 'bundle_name', 'bundle_id'], as_index=False)
    .agg({
        'total_rev': 'sum',
        'subscriptions': 'sum',
        'YEAR': 'first'
    })
)
weekly_purchases.shape

(26000882, 7)

In [27]:
weekly_purchases['WEEKNUMBER'] = weekly_purchases['WEEKNUMBER'].astype(int)

In [28]:
weekly_purchases.head(10)

,msisdn,WEEKNUMBER,bundle_name,bundle_id,total_rev,subscriptions,YEAR
0,242061000000,33,Forfait maximise 1.5GB 1 jours@650F,35054,650.0,1,2025
1,242061000000,33,Forfait weekend 3.17GB 3jours@840F,36826,840.0,1,2025
2,242061000000,34,Forfait maximise 1.5GB 1 jours@650F,35054,650.0,1,2025
3,242061000000,34,MagicNet 825MB 7 Jours@1200F,36965,1200.0,1,2025
4,242061000000,35,MagicNet 484MB 7 Jours@750F,36964,750.0,1,2025
5,242061000303,32,MaGicvoice 40+3Mins 7 jours@525F,FVX7D525FV4,525.0,1,2025
6,242061000404,31,Forfait Weekend 1.05GB 1 jours + 5 Mins@500F,36825,500.0,1,2025
7,242061000404,31,Forfait weekend 780MB 1jours@350F,36824,350.0,1,2025
8,242061000404,31,Ndeko_19Mins 1 jour@230F,80066,230.0,1,2025
9,242061000404,31,SMS Bundle 2 Days @66F,SMSPK50,66.0,1,2025


In [29]:
file_path = os.path.join(SAVE_TO, "weekly-purchase-202508.csv")
weekly_purchases.to_csv(file_path, index=False)

### Apply MBA ( we assume that weekly_purchases is table in MinTel Database schema)

In [16]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import col, expr, split, regexp_replace, trim, lit, concat, count, collect_list, isnan, when

In [8]:
# init spark
APP_NAME: str = "WEEKLY_MBA"

spark = SparkSession\
    .builder\
    .appName(APP_NAME)\
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")\
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")\
    .master("local[*]")\
    .config("spark.driver.memory", "16g")\
    .getOrCreate()

spark

In [9]:
DATA_PATH_W: str = r"C:\Users\Alber\Desktop\uni\GraduationProject\datasets\weekly purchases"
FILES_W: str = [rf'{DATA_PATH_W}\weekly-purchase-202508.csv',
            rf'{DATA_PATH_W}\weekly-purchase-202509.csv',
            rf'{DATA_PATH_W}\weekly-purchase-202510.csv'
]

In [10]:
data = spark.read.csv(
    FILES_W,
    header=True,
    inferSchema=True
).select("msisdn", "WEEKNUMBER", "bundle_name")

In [11]:
print((data.count(), len(data.columns)))

(69521293, 3)


In [13]:
data = data.filter(
    ~col("bundle_name").rlike("(?i)free|bonus|DIY")
)
data.count()

57476755

In [14]:
data.show(10, 0)

+------------+----------+--------------------------------------------+
|msisdn      |WEEKNUMBER|bundle_name                                 |
+------------+----------+--------------------------------------------+
|242061000000|33        |Forfait maximise 1.5GB 1 jours@650F         |
|242061000000|33        |Forfait weekend 3.17GB 3jours@840F          |
|242061000000|34        |Forfait maximise 1.5GB 1 jours@650F         |
|242061000000|34        |MagicNet 825MB 7 Jours@1200F                |
|242061000000|35        |MagicNet 484MB 7 Jours@750F                 |
|242061000303|32        |MaGicvoice 40+3Mins 7 jours@525F            |
|242061000404|31        |Forfait Weekend 1.05GB 1 jours + 5 Mins@500F|
|242061000404|31        |Forfait weekend 780MB 1jours@350F           |
|242061000404|31        |Ndeko_19Mins 1 jour@230F                    |
|242061000404|31        |SMS Bundle 2 Days @66F                      |
+------------+----------+--------------------------------------------+
only s

In [17]:
# Aggregate transactions per user per week
data = data.withColumn("transaction_id", concat(col("msisdn"), lit("_"), col("WEEKNUMBER")))
# trs = data.groupBy("msisdn", "WEEKNUMBER", "MONTHNUMBER").agg(collect_list("bundle_name").alias("bundle_names"))
trs = data.groupBy("transaction_id").agg(collect_list("bundle_name").alias("bundle_list"))

In [28]:
trs.show(10, 0)

+---------------+------------------------------------------------------------------------------------------------------------------------------------------+
|transaction_id |bundle_list                                                                                                                               |
+---------------+------------------------------------------------------------------------------------------------------------------------------------------+
|242061000000_35|[MagicNet 484MB 7 Jours@750F]                                                                                                             |
|242061000000_44|[MagicNet 825MB 7 Jours@1200F]                                                                                                            |
|242061000404_31|[Forfait Weekend 1.05GB 1 jours + 5 Mins@500F, Forfait weekend 780MB 1jours@350F, Ndeko_19Mins 1 jour@230F, SMS Bundle 2 Days @66F ]      |
|242061000505_35|[Forfait Maxivoice 16Mins 1 jour@200F]   

In [18]:
trs.count()

24014943

In [29]:
# Apply FP-Growth algorithm
min_support: float = 0.001 # Increase support to reduce rare itemsets
min_confidence: float = 0.5 # Increase confidence to keep only stronger rules
# Remove duplicates in the "bundle_name" array if exists.
trs = trs.withColumn("bundle_list", expr("array_distinct(bundle_list)"))
# run FP-Growth Algorithm
fp_growth = FPGrowth(itemsCol="bundle_list", 
                        minSupport=min_support,
                        minConfidence=min_confidence)
model = fp_growth.fit(trs)

In [24]:
model

FPGrowthModel: uid=FPGrowth_ba070799744e, numTrainingRecords=24014943

In [21]:
print("Frequent Itemsets Without Free Bundles:")
frq_itemssets = model.freqItemsets.show(n=100, truncate=False)
frq_itemssets
freq_rules_df = pd.DataFrame(model.freqItemsets.collect(), columns=model.freqItemsets.columns)
freq_rules_df.to_csv("no_free_bundles_freq_rules_weekly.csv", index=False)

Frequent Itemsets Without Free Bundles:
+---------------------------------------------------------------------------------------------------+-------+
|items                                                                                              |freq   |
+---------------------------------------------------------------------------------------------------+-------+
|[Forfait Maxivoice 16Mins 1 jour@200F]                                                             |4720129|
|[Forfait weekend 780MB 1jours@350F]                                                                |619515 |
|[Forfait Maxivoice 5Mins 1 jour@100F]                                                              |4213511|
|[Forfait Maxivoice 5Mins 1 jour@100F, Forfait Maxivoice 16Mins 1 jour@200F]                        |1158706|
|[MTN Call Me Back]                                                                                 |589922 |
|[SMS Bundle 1 Day @26F ]                                                       

In [30]:
print("Association Rules Without Free Bundles:")
association_rules = model.associationRules.show(n=100, truncate=False)
association_rules
association_rules_df = pd.DataFrame(model.associationRules.collect(), columns=model.associationRules.columns)
association_rules_df.to_csv("no_free_bundles_association_rules_weekly.csv", index=False)

Association Rules Without Free Bundles:
+-------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+------------------+------------------+---------------------+
|antecedent                                                                                                                                 |consequent                                    |confidence        |lift              |support              |
+-------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+------------------+------------------+---------------------+
|[100 SMS vers MTN ou 30 vers autre 3 jours@122F, SMS Bundle 3 Days@122F, Forfait Maxivoice 16Mins 1 jour@200F]                             |[SMS Bundle 2 Days @66F ]                     |0.5276287139746602|3.6680

In [ ]:
no_free_buundles_association_rules_df = pd.read_csv("no_free_bundles_association_rules_weekly.csv")
no_free_bundles_freq_rules_df = pd.read_csv("no_free_bundles_freq_rules_weekly.csv")
# clean duplicated " '' " in the consequent and antecedent
def clean_string_list(lst):
    return [item.replace("'", "") for item in lst]

no_free_buundles_association_rules_df['antecedent'] = no_free_buundles_association_rules_df['antecedent'].apply(lambda x: clean_string_list(ast.literal_eval(x)))
no_free_buundles_association_rules_df['consequent'] = no_free_buundles_association_rules_df['consequent'].apply(lambda x: clean_string_list(ast.literal_eval(x)))

# remove also duplicated " '' " from fre_rules
no_free_bundles_freq_rules_df['items'] = no_free_bundles_freq_rules_df['items'].apply(lambda x: clean_string_list(ast.literal_eval(x)))

In [ ]:
display(no_free_buundles_association_rules_df.head(20))
print(no_free_buundles_association_rules_df.shape)

In [ ]:
display(no_free_bundles_freq_rules_df.head(10))
print(no_free_bundles_freq_rules_df.shape)